# T11: satellite-made training photos through the unchanged pipeline

The team's rule: learned components are trained on satellite imagery only (no thermal photo). Block `train/checks/sat_photos.py` renders
drone-view photos (the official camera, level, height 100-350 m, random heading, metadata heading error up to +-10 deg) from a satellite image
**other than the pipeline's map** (Cartosat scene A): Cartosat scene B (another pass) and Pleiades Neo (another sensor), with thermal-like
changes (polarity flip 35 %, tone curve, blur, noise). Places: inside the pipeline's map window, more than ~1.5 km from every official photo
(exclusion box around the official photos), so no official place is trained on. Then the T6 rich-dump pipeline unchanged (27 queries, K = 50,
M = 10, raw matches) with a 5 km random hint, as for the dev photos. Geometry check (before this run): a photo rendered from scene A and
un-rendered by the pipeline's `make_query` correlates 0.93 / 0.87 with the map (0.15 / 0.22 when shifted by 15 m).
Env: N_B, N_PNEO, SEED, TAG.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
N_B, N_PNEO, SEED = int(os.environ.get("N_B", "300")), int(os.environ.get("N_PNEO", "100")), int(os.environ.get("SEED", "11"))
TAG = os.environ.get("TAG", "sat_train"); STRENGTH = [float(v) for v in os.environ.get("STRENGTH", "0").split(",")]; STRENGTH = STRENGTH[0] if len(STRENGTH) == 1 else tuple(STRENGTH)
OUT = ROOT / "tests/Test_Iteration_11/results"
from train.checks.rich_dump import RichDumpPipeline, pack_photo
pipeline = RichDumpPipeline("official", "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                            matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                            map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None,
                            headings=[-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0], scale_factors=[0.95, 1.0, 1.05])
t0 = time.time(); pipeline.setup(500, 7); print("setup %.0f s" % (time.time() - t0))

In [ ]:
from header import settings
from train.checks.sat_photos import Source, make_set
from evaluation.testset import TestSet
B = Source(ROOT / "dataset/Dataset_Official_2/268234921/BAND.tif")
PNEO = Source(next((ROOT / "dataset/small_sat_area").glob("*.tif")), bands=(1, 2, 3))
EXCLUDE = (296200, 301000, 2113760, 2118260)          # official photos' UTM box (297.7-299.5 km E, 2115.3-2116.8 km N) + 1.5 km
specs = [dict(name="B", source=B, n=N_B, box=(291700, 305500, 2109300, 2115250), exclude=EXCLUDE),
         dict(name="PNEO", source=PNEO, n=N_PNEO, box=(296150, 302750, 2114650, 2118780), exclude=EXCLUDE)]
items_df = make_set(str(OUT / "sat_photos" / TAG), specs, pipeline.camera, seed=SEED, geometry=dict(settings.GEOMETRY), strength=STRENGTH)
items = [dict(name=r["name"], lat=r["lat"], lon=r["lon"], rel_height=r["rel_height"], yaw=r["yaw"], pitch=0.0, roll=0.0, flight="sat")
         for _, r in items_df.iterrows()]
syn = TestSet(items, pipeline.camera, pipeline.satellite_map, np.array(["sat"] * len(items)), np.arange(len(items)))
items_df.groupby("source").agg(n=("name", "size"), h_min=("rel_height", "min"), h_max=("rel_height", "max"), flipped=("flip", "mean")).round(2)

In [ ]:
import torch
DUMP = OUT / "rich_dump" / TAG; DUMP.mkdir(parents=True, exist_ok=True)
done = {int(p.stem.split("_")[1]) for p in DUMP.glob("photo_*.npz")}
start = time.time(); failures = []
for index in [i for i in range(len(items)) if i not in done]:
    try:
        found = pipeline.run_queries(syn.items[index], syn.priors[index])
        arrays = pack_photo(syn.items[index], syn.truths[index], syn.sigmas[index], found["level"], found["queries"], min_keep_confidence=0.15)
        np.savez_compressed(DUMP / ("photo_%d.npz" % index), **arrays)
    except Exception as error:
        failures.append((index, repr(error)[:200])); print("photo", index, "failed", repr(error)[:200])
    if index % 25 == 0: print(index, "%.1f s per photo" % ((time.time() - start) / (index - len(done) + 1)), flush=True)
pd.DataFrame(failures, columns=["photo", "error"]).to_csv(DUMP / "failures.csv", index=False)
dict(photos=len(list(DUMP.glob("photo_*.npz"))), failed=len(failures), minutes=round((time.time() - start) / 60, 1))